# 19.17 Conformal prediction

Conformal prediction turns residual ranks into prediction sets with finite-sample coverage.

We split training, calibration, and test data, compute nonconformity scores, and use the ceil-correct quantile to build label sets with marginal coverage. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

Split conformal prediction includes labels whose score is below the calibration cutoff:

$$C(x)=\{y:s(x,y)\le q_{\lceil(n+1)(1-\alpha)\rceil}\}.$$

The lesson nonconformity components sum to $0.650$ and the guarded value is $0.715$.

In [ ]:

def conformal_sets(cal_scores, alpha, candidate_scores):
    n = len(cal_scores)
    rank = int(math.ceil((n + 1) * (1.0 - alpha)))
    rank = min(rank, n)
    qhat = float(np.sort(cal_scores)[rank - 1])
    sets = candidate_scores <= qhat
    return sets, qhat, rank

components = np.array([0.100, 0.200, 0.350])
total = float(components.sum())
abs_mass = float(np.abs(components).sum())
share = float(abs(components[0]) / abs_mass)
guarded = total + 0.1 * abs_mass
contrast = total - components[2]
change = contrast - total
sets, qhat, rank = conformal_sets(components, alpha=0.1, candidate_scores=np.array([[0.05, 0.40], [0.30, 0.20]]))

assert np.isclose(total, 0.650)
assert np.isclose(share, 0.154, atol=0.001)
assert np.isclose(guarded, 0.715)
assert np.isclose(change, -0.350)
print("total", total, "guard", guarded, "qhat", qhat, "rank", rank)


The reusable rung method trains a probabilistic classifier, uses $1-p_y$ as nonconformity, and measures coverage against the target $1-lpha$.

In [ ]:

def rung_conformal(X, y, alpha=0.1):
    clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te = fit_scaled_logistic_three_way(X, y)
    cal_proba = clf.predict_proba(x_cal)
    test_proba = clf.predict_proba(x_te)
    classes = clf.classes_
    cal_index = np.array([np.where(classes == label)[0][0] for label in y_cal])
    cal_scores = 1.0 - cal_proba[np.arange(len(y_cal)), cal_index]
    candidate_scores = 1.0 - test_proba
    prediction_sets, qhat, rank = conformal_sets(cal_scores, alpha, candidate_scores)
    test_index = np.array([np.where(classes == label)[0][0] for label in y_te])
    covered = prediction_sets[np.arange(len(y_te)), test_index]
    coverage = float(covered.mean())
    avg_size = float(prediction_sets.sum(axis=1).mean())
    gap = abs(coverage - (1.0 - alpha))
    return coverage, gap, avg_size, prediction_sets, y_te, qhat


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    coverage, gap, avg_size, sets, y_te, qhat = rung_conformal(X, y, alpha=0.1)
    rows.append((rung, name, coverage, gap, avg_size))

print("rung | coverage | gap | avg set size")
for rung, name, coverage, gap, avg_size in rows:
    print(f"D{rung} | {coverage:.3f} | {gap:.3f} | {avg_size:.2f} | {name}")


In [ ]:

summary = []
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for col, (name, X, y) in enumerate(rungs):
    coverage, gap, avg_size, sets, y_te, qhat = rung_conformal(X, y, alpha=0.1)
    sizes = sets.sum(axis=1)
    summary.append(gap)
    axes[col].hist(sizes, bins=np.arange(sizes.max() + 2) - 0.5, color="tab:green")
    axes[col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[col].set_xlabel("set size")
    axes[col].set_ylabel("count")

plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("coverage gap")
plt.title("Coverage gap vs. D1→D5")
plt.xticks(range(1, 6))
plt.show()


## Pitfall on D5: using the test set as calibration

If test labels choose the cutoff, the coverage claim is spent before evaluation. The fix uses a separate calibration split and the ceil-correct finite-sample rank.

In [ ]:

name, X, y = rungs[-1]
coverage, gap, avg_size, sets, y_te, qhat = rung_conformal(X, y, alpha=0.1)
clf, x_tr, x_cal, x_te, y_tr, y_cal, y_test = fit_scaled_logistic_three_way(X, y)
test_proba = clf.predict_proba(x_te)
classes = clf.classes_
test_index = np.array([np.where(classes == label)[0][0] for label in y_test])
leaky_scores = 1.0 - test_proba[np.arange(len(y_test)), test_index]
leaky_sets, leaky_qhat, leaky_rank = conformal_sets(leaky_scores, 0.1, 1.0 - test_proba)
leaky_coverage = float(leaky_sets[np.arange(len(y_test)), test_index].mean())

print("leaky test-calibrated coverage", leaky_coverage)
print("separate calibration coverage", coverage)
print("ceil-correct qhat", qhat)
print("leaky qhat", leaky_qhat)


## Evaluate it + Practice

- Metric: coverage gap from target 90% coverage.
- No-skill baseline: always return all labels.
- Cheap sanity check: with alpha 0.1, marginal coverage should be near 0.9.
- Ablation: replace ceil with floor and compare coverage.
- Failure signals: test-set calibration, subgroup gaps, or exploding set size.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Change alpha from 0.1 to 0.2 and track set sizes.

Practice 3: Compute coverage separately for the largest and smallest class.